In [1]:
# -*- coding: utf-8 -*-

# Sample Python code for youtube.channels.list
# See instructions for running these code samples locally:
# https://developers.google.com/explorer-help/code-samples#python

import os

import google_auth_oauthlib.flow
import googleapiclient.discovery
import googleapiclient.errors
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
import google.auth.exceptions
from google.auth.transport.requests import Request

import pickle
from convenient_pickle import *
import time
import datetime
import re

import country_converter as coco
from airports import airport_data
import pycountry

import datetime
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Please ensure that you've also downloaded the convenient_pickle library from Github. A lot of my code uses it to save non-dataframe data. You can store it in the same directory that this notebook is in.

In [2]:
#You'll need to change your data path to get this to work
data_path = '[your_data_path]'
current_directory = os.getcwd()
scopes = ["https://www.googleapis.com/auth/youtube.force-ssl"]

# handle_credentials() gets you a functioning youtube api session.  You'll need a client_secrets_file from youtube. Put that in whatever folder you have this in, and change the client_secrets_file to whatever your file is named.

# The first time you do this, it'll ask you to log in with whatever google account you got your api key from.

In [3]:
def handle_credentials():
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
    quota = 0
    api_service_name = "youtube"
    api_version = "v3"
    # Change the below to whatever your secrets file is called.
    client_secrets_file = "your_secrets_file"
    credentials = None
    if os.path.exists('token.json'):
        try:
            credentials = Credentials.from_authorized_user_file('token.json',scopes)
            credentials.refresh(Request())
        except google.auth.exceptions.RefreshError as error: 
            credentials = None
            print(f'{error}')
    if not credentials or not credentials.valid: 
        if credentials and credentials.expired and credentials.refresh_token: 
            credentials.refresh(Request())
        else: 
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                client_secrets_file, scopes)
            credentials = flow.run_local_server(port=0)
    with open('token.json', 'w') as token: 
        token.write(credentials.to_json())
    youtube = googleapiclient.discovery.build(
        api_service_name, api_version, credentials=credentials
        )
    return youtube

get_top_x_from_query has 5 parts.

1. Query. This is what you want to search for.
2. Before. This is the date that you want to search before.
3. After. This is the date that you want to search after.
4. adddon. This will let you add a term onto the default query so that it won't affect the resulting keys.
5. query_range. This is how many pages' worth of query you want to go through. 1 page is 50 results (mostly; not all of the urls that come back are functional videos). Remember that 1 page also costs 100 units of quota, so this should be used sparingly.

Dates take the format of "YYYY-MM-DDTHH\:MM\:SSZ". Examples are provided as default values in the input parameters for this.

You'll receive a list of youtube search results (including video ids, which is what we're after). This will be split up by query. If my query is "Slovenia" that would be a key. If I want to search "travel Slovenia" but keep the key as "Slovenia", I could just set "addon" to "travel"

In [4]:
def get_top_x_from_query(query, before='2025-12-9T00:00:00Z',after='2024-06-01T00:00:00Z', addon='travel', query_range=1):
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
    quota = 0
    youtube = handle_credentials()
    outlist = []
    next_page_token = None
    for _ in range(query_range):
        search_request = youtube.search().list(
            part="snippet",
            maxResults=50,
            publishedAfter=after,
            publishedBefore=before,
            q=f"{addon} {query}",
            pageToken=next_page_token
        )
        result = search_request.execute()
        outlist.append(result)
        next_page_token = result.get("nextPageToken")
    return outlist

This will give you a list of country codes. I made it so that Kosovo's country code returns Kosovo's name, that Great Britain returns the countries of Great Briain (England, Scotland, Wales, etc), and that Moldova, Bosnia and Macedonia have more usable/searchable names (by default Bosnia gets used as "Bosnia and Herzegovina" for instance). I also use both Türkiye and Turkey, because a number of videos might use either spelling.

In [17]:
# Create a list of country topics to go through

country_codes = ['PT', 'ES', 'NO', 'XK', 'TR', 'RO', 'FR', 'FI', 'GR',
             'BG', 'HU', 'IT', 'SE', 'NL', 'SK', 'DK', 'SI', 'LT',
             'CY', 'GB', 'AM', 'PL', 'CZ', 'IE', 'EE', 'LV', 'HR',
             'GE', 'MT', 'RS', 'BA', 'MK', 'AL', 'LU', 'MD', 'IS', 
             'ME', 'AZ']

country_names = []

for code in country_codes: 
    if code =='XK':
        country_names.append("Kosovo")
    elif code =="GB": 
        country_names.append("United Kingdom")
    elif code == "MD": 
        country_names.append("Moldova")
    elif code == "BA": 
        country_names.append("Bosnia")
    elif code == 'MK':
        country_names.append("Macedonia")
    else: 
        outcountry = pycountry.countries.search_fuzzy(code)[0]
        country_names.append(outcountry.name)

#There was an issue with scraping these the first time, so they're manually labeled

german_country_names = ['Austria','Belgium','Switzerland','Germany']
country_names+= german_country_names


#Alternatively, if you only want to search 1 or a handful of countries, feel free to uncomment one of the below

#country_names = german_country_names
country_names = ["Albania"]


tagnames = ["travel "+name.lower() for name in country_names]
zippednames = [i for i in zip(country_names,tagnames)]

# This will go through each country in the list and add it to a pickled dictionary. By default, it is commented out so you don't accidentally redo it multiple times and burn through your quota.

# You can change the "start" and "end" variables to determine how many you scrape. By default, just leave as-is. 

# Bear in mind that each scraping process eats up 100 quota units of Youtue API quota, so if you only have the default 10,000 units, scraping all of the countries will exhaust your quota very quickly. 

In [14]:
#Kira: [0-13]
#Himanshu: [13:26]
#Sam: [26:]

#replace these with the proper numbers depending on the day. 1-3, 2-4, etc. 
start = 0
end = 2

for zipname in zippednames[start:end]: 
    use_name = zipname[0]
    country_list = get_top_x_from_query(use_name,addon='travel',query_range=50)
    dump_pickle(os.getcwd(),f'/country_pickle_files/youtube_travel_top_100_{use_name}.pkl',country_list,warn=False)

('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=188680021294-khdqb4lu47044hu195cap17hvfo2fq04.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A57341%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.force-ssl&state=2nv9ToJvH3WNgwF0GzibgwMDclXjDt&access_type=offline


In [30]:
print(zipname)

('Austria', 'travel austria')


In [11]:
', '.join([i[0] for i in zippednames[13:26]])

'Netherlands, Slovakia, Denmark, Slovenia, Lithuania, Cyprus, United Kingdom, Armenia, Poland, Czechia, Ireland, Estonia, Latvia'

In [34]:
', '.join([i[0] for i in zippednames[26:]])

'Croatia, Georgia, Malta, Serbia, Bosnia, Macedonia, Albania, Luxembourg, Moldova, Iceland, Montenegro, Azerbaijan'

In [18]:
zippednames[start:end]

[('Albania', 'travel albania')]